# TNG tour
A hands-on tour of the TNG module (`galactic_tidal_constraints.tng`): it reads the IllustrisTNG (TNG50-1) data this
project already has, exactly as stored, with no unit changes and no change of frame. Run the cells in order.

- `tng.read_table(path)`: the target table, one array per column
- `tng.read_archive(path, particle_types)`: one target's particles

The files live in the data folder, outside the repo. The tour looks for it in `GTC_DATA_DIR`, or else in `data/`
next to the repo. Design: `docs/modules/tng.md`.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints import tng

DATA = Path(os.environ.get("GTC_DATA_DIR", "../../../data")) / "tng-data"
TARGET = 62000000190717

## The target table
`read_table` returns a plain dict: column name → array, one entry per row.

In [ ]:
table = tng.read_table(DATA / "moi-4-targets.h5")

print(type(table).__name__, "with", len(table), "columns,", len(table["snap"]), "rows")
print("encounters:", len(np.unique(table["moi_2"])))

kinds = {}
for name, column in table.items():
    kinds.setdefault((column.dtype.kind, column.shape[1:]), []).append(name)
for (kind, shape), names in kinds.items():
    print()
    print(f"{len(names):3d} columns of dtype kind {kind!r}, shape (rows, *{shape}):")
    print("    " + ", ".join(names[:6]) + (" ..." if len(names) > 6 else ""))

### One target's row
IDs are integers, so a full `SubhaloIDRaw` matches exactly. The catalog values are raw TNG units (ckpc/h, km/s, 10¹⁰ M☉/h).

In [ ]:
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
for name in ("moi_2", "snap", "p_SubhaloIDRaw", "s_SubhaloIDRaw", "redshift", "scale_factor", "time_gyr",
             "p_SubhaloMass", "s_SubhaloMass", "p_SubhaloPos", "s_SubhaloPos", "p_SubhaloVel", "p_SubhaloSpin"):
    print(f"{name:16s} {table[name][row]}")
print()
print("orbital_frame_pos_transform:")
print(np.round(table["orbital_frame_pos_transform"][row], 3))

### One encounter's rows
All snapshots of the same encounter share `moi_2`.

In [ ]:
rows = np.flatnonzero(table["moi_2"] == table["moi_2"][row])
rows = rows[np.argsort(table["snap"][rows])]
print("snapshots:", table["snap"][rows].tolist())
print("time (Gyr): %.2f to %.2f" % (table["time_gyr"][rows][0], table["time_gyr"][rows][-1]))

separation = np.linalg.norm(table["s_SubhaloPos"][rows] - table["p_SubhaloPos"][rows], axis=1)
plt.figure(figsize=(6, 3))
plt.plot(table["time_gyr"][rows], separation, "o-")
plt.axvline(table["time_gyr"][row], color="gray", ls=":", label="target snapshot")
plt.xlabel("age of the universe (Gyr)"); plt.ylabel("separation (ckpc/h, as stored)"); plt.legend();

## A target archive
Stars by default. The values are raw TNG snapshot units, in the earlier pipeline's orbital frame.

In [ ]:
stars = tng.read_archive(DATA / f"target-archive-{TARGET}.h5")
for name, value in stars.items():
    print(f"{name:18s}", f"{value.shape} {value.dtype}" if isinstance(value, np.ndarray) else value)

In [ ]:
for name in ("subfind_galaxy", "historical_galaxy"):
    print(f"{name:18s} primary {int((stars[name] == 0).sum()):,}   secondary {int((stars[name] == 1).sum()):,}")
moved = int((stars["subfind_galaxy"] != stars["historical_galaxy"]).sum())
print(f"stars whose historical galaxy differs from SUBFIND's: {moved:,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
for ax, galaxy, title in zip(axes, (0, 1), ("primary", "secondary")):
    xy = stars["positions"][stars["historical_galaxy"] == galaxy, :2]
    ax.hist2d(xy[:, 0], xy[:, 1], bins=300, range=[[-60, 60], [-60, 60]], norm="log", cmap="gray")
    ax.set_title(f"{title} stars (historical label)"); ax.set_aspect("equal"); ax.set_xlabel("x (ckpc/h, as stored)")
axes[0].set_ylabel("y (ckpc/h, as stored)");

### Other particle types
One name, several, or `"all"`.

In [ ]:
everything = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "all")
for name, number in tng.PARTICLE_TYPES.items():
    kept = everything["particle_type"] == number
    total = everything["masses"][kept].sum(dtype=np.float64)
    print(f"{name:6s} {int(kept.sum()):>10,}   total mass {total:10.3f}  (1e10 Msun/h)")

dm = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "dm")
print()
print("dark matter particle mass:", np.unique(dm["masses"]), "=", f"{dm['masses'][0] * 1e10 / 0.6774:.3e} Msun (TNG50-1: 4.5e5)")

## Measuring a galaxy
`galaxy_properties` takes plain arrays and returns the mass, center of mass, mean velocity and spin (angular
momentum per unit mass). Nothing is converted: the results are in the units of the arrays given.

In [ ]:
archive = tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "all")
measured = {}
for galaxy, side in ((0, "p"), (1, "s")):
    kept = archive["subfind_galaxy"] == galaxy
    measured[side] = tng.galaxy_properties(archive["positions"][kept], archive["velocities"][kept], archive["masses"][kept])

for name, value in measured["p"].items():
    print(f"{name:15s} {type(value).__name__:8s} {value}")

### Against the TNG catalog
The same two galaxies in the catalog (the table's `p_` and `s_` columns). With a the snapshot's scale factor:
velocities need √a, and the catalog's spin is the particles' spin × a^1.5 (a physical length and a physical velocity).

In [ ]:
table = tng.read_table(DATA / "moi-4-targets.h5")
row = int(np.flatnonzero(table["p_SubhaloIDRaw"] == TARGET)[0])
a = table["scale_factor"][row]
size = np.linalg.norm

print(f"a = {a:.4f}\n")
print(f"{'':32s}{'particles':>12s}{'catalog':>12s}{'ratio':>10s}")
def line(label, mine, catalog):
    print(f"{label:32s}{mine:12.4f}{catalog:12.4f}{mine / catalog:10.5f}")

for side, name in (("p", "primary"), ("s", "secondary")):
    spin, catalog_spin = size(measured[side]["spin"]), size(table[f"{side}_SubhaloSpin"][row])
    line(f"{name} mass", measured[side]["mass"], table[f"{side}_SubhaloMass"][row])
    line(f"{name} spin, as stored", spin, catalog_spin)
    line(f"{name} spin x a^1.5", spin * a**1.5, catalog_spin)
    line(f"{name} spin x mass (the old way)", spin * measured[side]["mass"], catalog_spin)

separation = size(measured["p"]["center_of_mass"] - measured["s"]["center_of_mass"])
speed = size(measured["p"]["velocity"] - measured["s"]["velocity"])
line("separation of the centers", separation, size(table["p_SubhaloCM"][row] - table["s_SubhaloCM"][row]))
line("relative speed, as stored", speed, size(table["p_SubhaloVel"][row] - table["s_SubhaloVel"][row]))
line("relative speed x sqrt(a)", speed * np.sqrt(a), size(table["p_SubhaloVel"][row] - table["s_SubhaloVel"][row]))

### Through the old frame
The archive's particles are in the earlier pipeline's frame: the box moved by the table's two 4×4 matrices (a rotation
and a shift). Positions and velocities take the whole matrix; a spin takes the rotation only.

In [ ]:
pos_matrix, vel_matrix = table["orbital_frame_pos_transform"][row], table["orbital_frame_vel_transform"][row]
rotation = vel_matrix[:3, :3]
move = lambda matrix, vector: matrix[:3, :3] @ vector + matrix[:3, 3]
angle = lambda u, v: np.degrees(np.arccos(np.clip(u @ v / size(u) / size(v), -1, 1)))

print("the two matrices share one rotation:", np.array_equal(pos_matrix[:3, :3], rotation),
      "  det =", round(float(np.linalg.det(rotation)), 6))
for side, name in (("p", "primary"), ("s", "secondary")):
    center = move(pos_matrix, table[f"{side}_SubhaloCM"][row])
    velocity = move(vel_matrix, table[f"{side}_SubhaloVel"][row] / np.sqrt(a))
    catalog_spin = table[f"{side}_SubhaloSpin"][row]
    print(f"\n{name}")
    print(f"  center of mass    particles {np.round(measured[side]['center_of_mass'], 3)}   catalog, moved {np.round(center, 3)}")
    print(f"  velocity          particles {np.round(measured[side]['velocity'], 3)}   catalog / sqrt(a), moved {np.round(velocity, 3)}")
    print(f"  spin direction    off by {angle(measured[side]['spin'], rotation @ catalog_spin):.4f} deg with the rotation only,"
          f" {angle(measured[side]['spin'], move(vel_matrix, catalog_spin)):.1f} deg with the shift too (the old way)")

### Half-mass radius
`half_mass_radius` measures from the center it is given. The catalog measures from `SubhaloPos` (the most bound
particle), not from the center of mass.

In [ ]:
for galaxy, side, name in ((0, "p", "primary"), (1, "s", "secondary")):
    kept = archive["subfind_galaxy"] == galaxy
    most_bound = move(pos_matrix, table[f"{side}_SubhaloPos"][row])
    all_types = tng.half_mass_radius(archive["positions"][kept], archive["masses"][kept], most_bound)
    stars = kept & (archive["particle_type"] == tng.PARTICLE_TYPES["stars"])
    stellar = tng.half_mass_radius(archive["positions"][stars], archive["masses"][stars], most_bound)
    print(f"{name:10s} all types {all_types:8.4f}  catalog {table[f'{side}_SubhaloHalfmassRad'][row]:8.4f}"
          f"   stars only {stellar:7.4f}   (ckpc/h, as stored)")

## Errors
A wrong particle type or a missing file is named in the error.

In [ ]:
for call in (lambda: tng.read_archive(DATA / f"target-archive-{TARGET}.h5", "wind"),
             lambda: tng.read_archive(DATA / "target-archive-1.h5"),
             lambda: tng.read_table(DATA / "nothing.h5")):
    try:
        call()
    except (ValueError, FileNotFoundError) as error:
        print(type(error).__name__ + ":", error)